In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import os

In [2]:
def extract_atom_counts(file_path):
    atom_counts = {}
    inside_atom_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Atom types and atom numbers"):
                inside_atom_section = True
                continue
            if inside_atom_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 2:
                    atom, count = parts
                    atom_counts[atom] = int(count)
    
    return atom_counts

In [3]:
def extract_lattice_vectors(file_path):
    lattice_vectors = []
    inside_lattice_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Primitive lattice vectors (ang.)"):
                inside_lattice_section = True
                continue
            if inside_lattice_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3:
                    vector = list(map(float, parts))
                    lattice_vectors.append(vector)
    
    return lattice_vectors

In [4]:
def extract_atomic_positions(file_path):
    atomic_positions = []
    inside_positions_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Atomic positions (ang.)"):
                inside_positions_section = True
                continue
            if inside_positions_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3:
                    position = list(map(float, parts))
                    atomic_positions.append(position)
    
    return atomic_positions

In [5]:
def extract_forces(file_path):
    force = []
    inside_force_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Forces (eV ang.^-1)"):
                inside_force_section = True
                continue
            if inside_force_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3:
                    position = list(map(float, parts))
                    force.append(position)
    
    return force

In [6]:
def is_float(value):
    try:
        float(value)
        return True
    except ValueError:
        return False

def extract_stress(file_path):
    stress = []
    inside_stress_section = False

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Stress (kbar)"):
                inside_stress_section = True
                continue
            if inside_stress_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 3 and is_float(parts[0]):
                    position = list(map(float, parts))
                    stress.append(position)
    
    return stress

In [7]:
def extract_total_energy(file_path):
    inside_energy_section = False
    energy = []

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Total energy (eV)"):
                inside_energy_section = True
                continue
            if inside_energy_section:
                if line.startswith("="):
                    break
                parts = line.split()
                if len(parts) == 1 and is_float(parts[0]):
                    position = list(map(float, parts))
                    energy.append(position)
    
    return energy

In [8]:
import os

def extract_and_save_sections(file_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    sections = {}
    current_section = None
    current_content = []

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("Configuration num."):
                if current_section is not None:
                    config_num = current_section.split()[-1]
                    file_name = f"ML_ABN_{config_num}.txt"
                    file_path = os.path.join(output_dir, file_name)
                    with open(file_path, 'w') as section_file:
                        section_file.write("\n".join(current_content))
                    sections[config_num] = file_path
                    current_content = []
                current_section = line
                continue
            if current_section is not None:
                current_content.append(line)
        
        if current_section is not None:
            config_num = current_section.split()[-1]
            file_name = f"ML_ABN_{config_num}.txt"
            file_path = os.path.join(output_dir, file_name)
            with open(file_path, 'w') as section_file:
                section_file.write("\n".join(current_content))
            sections[config_num] = file_path

    return sections

In [12]:
def write_train_cfg(sections, output_dir):
    atom_type_mapping = {'C': 0, 'Mn': 1, 'Fe': 2, 'N': 3, 'Na': 4, 'O': 5, 'H': 6}

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    for config_num, file_path in sections.items():
        atom_counts  = extract_atom_counts(file_path)
        lattice_vectors = extract_lattice_vectors(file_path)
        atomic_positions = extract_atomic_positions(file_path)
        forces = np.array(extract_forces(file_path))
        total_energy = extract_total_energy(file_path)[0][0]
        stress_tensor = extract_stress(file_path)

        atom_data = []
        atom_type_list = []
        for atom_type, count in atom_counts.items():
            atom_type_list.extend([atom_type_mapping[atom_type]] * count)
            
        file_name = f"train.cfg_{config_num}.txt"
        cfg_file_path = os.path.join(output_dir, file_name)
        
        with open(cfg_file_path, 'w') as file:
            file.write("BEGIN_CFG\n")
            file.write(f" Size\n")
            file.write(f"    {len(atomic_positions)}\n")
            file.write(" Supercell\n")
            for vec in lattice_vectors:
                file.write(f"    {'    '.join(f'{v:15.8f}' for v in vec)}\n")
            file.write(" AtomData:  id type       cartes_x      cartes_y      cartes_z           fx          fy          fz\n")
            for i, (pos, force) in enumerate(zip(atomic_positions, forces)):
                atom_type = atom_type_list[i]
                file.write(f"    {i+1:<3} {atom_type:<4} {'    '.join(f'{p:15.8f}' for p in pos)}    {'    '.join(f'{f:15.8f}' for f in force)}\n")
            file.write(f" Energy\n")
            file.write(f"    {total_energy:15.6f}\n")
            if stress_tensor:
                file.write(" PlusStress:  xx          yy          zz          yz          xz          xy\n")
                for tensor in stress_tensor:
                    file.write(f"    {'    '.join(f'{t:15.8f}' for t in tensor)}")
            file.write("\n Feature   EFS_by       VASP\n")
            file.write("END_CFG\n\n")

In [13]:
file_path = f'./ML_AB'
output_dir = 'training_set'
sections = extract_and_save_sections(file_path, output_dir)
cfg_output_dir = f'training_set'
write_train_cfg(sections, cfg_output_dir)
print(f"train.cfg files have been written to directory: {cfg_output_dir}")


train.cfg files have been written to directory: training_set


In [13]:
# Example usage:
cfg_output_dir = './train.cfg'
write_train_cfg(sections, cfg_output_dir)

print(f"train.cfg files have been written to directory: {cfg_output_dir}")


train.cfg files have been written to directory: ./train.cfg


In [10]:
file_path = f'./ML_ABN_val'
output_dir = f'./validation'
sections = extract_and_save_sections(file_path, output_dir)
cfg_output_dir = f'./validation'
write_train_cfg(sections, cfg_output_dir)
print(f"train.cfg files have been written to directory: {cfg_output_dir}")

train.cfg files have been written to directory: ./validation


In [21]:
# Example usage:
file_path = './ML_ABN_format.txt'
#atom_counts, lattice_vectors, atomic_positions, forces, stress, energy= extract_structure(file_path)
atom_counts = extract_atomic_positions(file_path)
lattice_vectors =  np.array(extract_lattice_vectors(file_path))
atomic_positions =  np.array(extract_atomic_positions(file_path))
forces = np.array(extract_forces(file_path))
stress =  np.array(extract_stress(file_path))
total_energy = extract_total_energy(file_path)

In [41]:
total_energy = extract_total_energy(file_path)
total_energy[0][0]

-996.30333699251

In [21]:
atom_list = ['C', 'Mn', 'Fe', 'N', 'Na']

In [27]:
atom_counts = extract_atom_counts(file_path)
for i in range(len(atom_counts)):
    print(i)
    atom_numbers = [] 
    atom_numbers.append(atom_counts[atom_list[i]])
    
sum(atom_numbers)

0
1
2
3
4


16

In [28]:
atom_numbers = sum(atom_counts.values())
atom_numbers

128

In [29]:
atom_counts

{'C': 48, 'Mn': 8, 'Fe': 8, 'N': 48, 'Na': 16}